# Trabajo Guia de Laboratorio 01

In [39]:
#importamos librerias

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [40]:
#cargamos el data frame y mostramos la cantidad de datos
db_path = os.path.join('BD', 'DETALLE_INVERSIONES.csv')
df = pd.read_csv(db_path, low_memory = False)

filas, columnas = df.shape
print("Filas:", filas)
print("Columnas:", columnas)

Filas: 260009
Columnas: 68


### 4 . PERFILADO

#### NIVEL 1 — Identidad: Validación del Código Único (CUI)
Antes de procesar cualquier dato financiero o técnico, debemos asegurar que el registro represente un proyecto real y rastreable en el sistema Invierte.pe.

**Lógica del Perfilado:**
* `mask_cui_invalido`: Identifica registros donde el `CODIGO_UNICO` es nulo (vacío) o no cumple con el estándar técnico de 7 dígitos (rango entre 1,000,000 y 9,999,999).
* `error_CU`: Almacena los registros descartados. Si el ID está mal, el registro no tiene trazabilidad.
* `df_valido`: Es nuestro "dataset maestro" filtrado que pasará al siguiente nivel.

**Variables involucradas:**
* `CODIGO_UNICO`: Identificador numérico clave.

In [41]:
# Convertimos a numérico para validar rangos
df['CODIGO_UNICO'] = pd.to_numeric(df['CODIGO_UNICO'], errors='coerce')

# Definimos las condiciones de error
mask_cui_invalido = df['CODIGO_UNICO'].isna() | ~df['CODIGO_UNICO'].between(1000000, 9999999)

# Separamos la data
error_CU = df[mask_cui_invalido].copy()
df_valido = df[~mask_cui_invalido].copy()

print(f"Nivel 1 finalizado:")
print(f"- Registros inválidos (error_CU): {len(error_CU)}")
print(f"- Registros con identidad confirmada (df_valido): {len(df_valido)}")

Nivel 1 finalizado:
- Registros inválidos (error_CU): 67
- Registros con identidad confirmada (df_valido): 259942


#### NIVEL 2 — Lógica de Negocio: Estado vs. Situación
En este nivel verificamos la coherencia administrativa. Un proyecto no puede estar "operando" si legalmente no ha sido aprobado.

**Lógica del Perfilado:**
* `mask_estado_incoherente`: Detecta proyectos que figuran como **ACTIVO** en su estado, pero cuya situación técnica no es **VIABLE**. En la gestión pública, la viabilidad es el "permiso legal" indispensable para ejecutar recursos.
* `error_estado`: Captura estas inconsistencias donde el sistema muestra actividad en proyectos no autorizados.

**Variables involucradas:**
* `ESTADO`: Situación administrativa (Activo, Desactivado, etc.).
* `SITUACION`: Condición técnica (Viable, Aprobado, En Formulación).

In [42]:
# Normalizamos textos para evitar errores por espacios o minúsculas
df_valido['ESTADO'] = df_valido['ESTADO'].astype(str).str.strip().str.upper()
df_valido['SITUACION'] = df_valido['SITUACION'].astype(str).str.strip().str.upper()

# Filtro de inconsistencia: Activo pero no Viable
mask_estado_incoherente = (df_valido['ESTADO'] == 'ACTIVO') & (df_valido['SITUACION'] != 'VIABLE')

error_estado = df_valido[mask_estado_incoherente].copy()
df_proyectos_coherentes = df_valido[~mask_estado_incoherente].copy()

print(f"Nivel 2 finalizado:")
print(f"- Proyectos con estado incoherente: {len(error_estado)}")
print(f"- Proyectos con lógica de negocio aprobada: {len(df_proyectos_coherentes)}")

Nivel 2 finalizado:
- Proyectos con estado incoherente: 59843
- Proyectos con lógica de negocio aprobada: 200099


#### NIVEL 3 — Línea de Tiempo: Cronología del Proyecto
Validamos que el ciclo de vida del proyecto siga un orden cronológico lógico. El tiempo en la inversión pública es lineal y sucesivo.

**Lógica del Perfilado:**
* `error_fechas`: Registros donde la `FECHA_REGISTRO` es posterior a la `FECHA_VIABILIDAD`. Es imposible aprobar un proyecto antes de que sea registrado como idea.
* También se evalúa que la ejecución (`FEC_INI_EJECUCION`) no haya iniciado antes de obtener la viabilidad.

**Variables involucradas:**
* `FECHA_REGISTRO`: Nacimiento de la idea.
* `FECHA_VIABILIDAD`: Aprobación técnica/económica.
* `FEC_INI_EJECUCION`: Inicio de obra física.

In [43]:
# Conversión masiva a formato fecha
for col in ['FECHA_REGISTRO', 'FECHA_VIABILIDAD', 'FEC_INI_EJECUCION']:
    df_proyectos_coherentes[col] = pd.to_datetime(df_proyectos_coherentes[col], errors='coerce')

# Validación de orden lógico
mask_cronologia_erronea = (df_proyectos_coherentes['FECHA_REGISTRO'] > df_proyectos_coherentes['FECHA_VIABILIDAD']) | \
                          (df_proyectos_coherentes['FEC_INI_EJECUCION'] < df_proyectos_coherentes['FECHA_VIABILIDAD'])

error_cronologia = df_proyectos_coherentes[mask_cronologia_erronea].copy()

print(f"Nivel 3 finalizado:")
print(f"- Proyectos con fechas imposibles: {len(error_cronologia)}")

Nivel 3 finalizado:
- Proyectos con fechas imposibles: 8091


#### NIVEL 4 — Consistencia Financiera: Análisis de Saldos
Analizamos si los montos de dinero pendientes de ejecutar tienen coherencia con el presupuesto total asignado al proyecto.

**Lógica del Perfilado:**
* `error_financiero_critico`: Ocurre cuando el `SALDO_EJECUTAR` es mayor al `COSTO_ACTUALIZADO`. No puedes tener pendiente más dinero del que cuesta el proyecto total.
* `advertencia_sobrecosto`: Cuando el saldo pendiente ya superó el `MONTO_VIABLE` original, indicando que el proyecto se ha encarecido más de lo previsto inicialmente.

**Variables involucradas:**
* `SALDO_EJECUTAR`: Dinero que falta gastar.
* `COSTO_ACTUALIZADO`: Valor total del proyecto hoy.
* `MONTO_VIABLE`: Valor original aprobado.

In [44]:
# Aseguramos formato numérico en montos
for col in ['SALDO_EJECUTAR', 'COSTO_ACTUALIZADO', 'MONTO_VIABLE']:
    df_proyectos_coherentes[col] = pd.to_numeric(df_proyectos_coherentes[col], errors='coerce')

# Detección de errores críticos (Saldo > Total)
mask_financiero_critico = df_proyectos_coherentes['SALDO_EJECUTAR'] > df_proyectos_coherentes['COSTO_ACTUALIZADO']
error_financiero = df_proyectos_coherentes[mask_financiero_critico].copy()

# Detección de advertencias (Saldo > Viable inicial)
mask_sobrecosto = df_proyectos_coherentes['SALDO_EJECUTAR'] > df_proyectos_coherentes['MONTO_VIABLE']
advertencia_financiera = df_proyectos_coherentes[mask_sobrecosto].copy()

print(f"Nivel 4 finalizado:")
print(f"- Errores financieros críticos: {len(error_financiero)}")
print(f"- Alertas por sobrecostos detectados: {len(advertencia_financiera)}")

Nivel 4 finalizado:
- Errores financieros críticos: 1
- Alertas por sobrecostos detectados: 26323


#### NIVEL 5 — Realidad Física: Porcentajes de Avance
Finalmente, verificamos los indicadores de progreso. Los porcentajes son valores matemáticos acotados.

**Lógica del Perfilado:**
* `error_avance`: Identifica valores fuera del rango [0 - 100]. 
* **Nota técnica:** Valores extremadamente altos (como 200,000,000) sugieren que el usuario ingresó el monto ejecutado en soles en la celda destinada al porcentaje (error de campo).

**Variables involucradas:**
* `AVANCE_FISICO`: Progreso de la obra en el campo.
* `AVANCE_EJECUCION`: Progreso del gasto financiero.

In [45]:
# Validación de rango porcentual [0-100]
df_proyectos_coherentes['AVANCE_FISICO'] = pd.to_numeric(df_proyectos_coherentes['AVANCE_FISICO'], errors='coerce')

mask_avance_imposible = (df_proyectos_coherentes['AVANCE_FISICO'] < 0) | (df_proyectos_coherentes['AVANCE_FISICO'] > 100)
error_avance = df_proyectos_coherentes[mask_avance_imposible].copy()

print(f"Nivel 5 finalizado:")
print(f"- Registros con porcentajes imposibles: {len(error_avance)}")
if len(error_avance) > 0:
    print(f"- Valor máximo detectado erróneamente: {error_avance['AVANCE_FISICO'].max()}")

Nivel 5 finalizado:
- Registros con porcentajes imposibles: 140
- Valor máximo detectado erróneamente: 61325049.14


#### NIVEL 6 — Resumen de Hallazgos: Impacto en la Calidad de Datos
En este nivel final, consolidamos todas las métricas de error detectadas para obtener una visión general de la salud del dataset. Este paso es crucial para determinar si la información es apta para la toma de decisiones estratégicas en el proyecto.

**Lógica del Perfilado:**
* **Consolidación:** Agrupamos los conteos de registros afectados en cada nivel previo (Identidad, Lógica, Cronología, Financiero y Física).
* **Cálculo de Representatividad:** Determinamos el `% del total` para cada incidencia. Esto permite priorizar qué errores requieren una limpieza inmediata.
* `resumen_perfilado`: Genera una tabla técnica con las categorías y descripciones de los fallos encontrados.


**Variables involucradas:**
* `resumen_perfilado`: DataFrame que centraliza los resultados de los 5 niveles técnicos previos.
* `% del total`: Cálculo de representatividad que mide el impacto de cada inconsistencia sobre el universo total de datos.
* `total`: Variable de control que almacena el número total de registros procesados.

In [46]:

total = len(df)

# Creamos la estructura del reporte final basada en los niveles previos
resumen_perfilado = pd.DataFrame([
    (1, 'Identidad',          'CUI nulo o sin 7 dígitos',              len(error_CU)),
    (2, 'Lógica de Negocio',   'Activo sin Situación Viable',           len(error_estado)),
    (3, 'Cronología',          'Fechas fuera de orden lógico',          len(error_cronologia)),
    (4, 'Financiero - Crítico','Saldo > Costo actualizado',             len(error_financiero)),
    (4, 'Financiero - Alerta', 'Saldo > Monto viable',                  len(advertencia_financiera)),
    (5, 'Realidad Física',     'Avance fuera de rango (0-100)',         len(error_avance)),
], columns=['Nivel', 'Categoría', 'Descripción', 'Registros_afectados'])

# Cálculo del impacto porcentual
resumen_perfilado['% del total'] = (
    resumen_perfilado['Registros_afectados'] / total * 100
).round(2)

# Impresión del reporte técnico en formato plano
print("=====================================================================")
print("           REPORTE FINAL DE PERFILADO DE DATOS                       ")
print("=====================================================================")
print(resumen_perfilado.to_string(index=False))
print("=====================================================================")
print(f"Total de registros procesados: {total}")

           REPORTE FINAL DE PERFILADO DE DATOS                       
 Nivel            Categoría                   Descripción  Registros_afectados  % del total
     1            Identidad      CUI nulo o sin 7 dígitos                   67         0.03
     2    Lógica de Negocio   Activo sin Situación Viable                59843        23.02
     3           Cronología  Fechas fuera de orden lógico                 8091         3.11
     4 Financiero - Crítico     Saldo > Costo actualizado                    1         0.00
     4  Financiero - Alerta          Saldo > Monto viable                26323        10.12
     5      Realidad Física Avance fuera de rango (0-100)                  140         0.05
Total de registros procesados: 260009


### 5 . LIMPIEZA DE DATOS